# RAG evaluation on a natural questions dataset

This notebook is a modified version of https://github.com/qingyun-wu/autogen-eval/blob/main/application/A2-retrieval-augmented-chat/NaturalQuestionsQA-gpt35turbo.ipynb

In [7]:
import os
from wrappers import RagWrapper, RagNoInteractionWrapper

config_list = [{"model": "gpt-3.5-turbo", "api_key": os.environ["OPENAI_API_KEY"]}]
rag_no_interaction_wrapper = RagNoInteractionWrapper(config_list)
rag_wrapper = RagWrapper(config_list)

/Users/nikitakostin/PycharmProjects/lab-070524/.venv/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/Users/nikitakostin/PycharmProjects/lab-070524/.venv/lib/python3.9/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [17]:
import json
import random

queries_file = "https://huggingface.co/datasets/thinkall/NaturalQuestionsQA/resolve/main/queries.jsonl"

!wget -O /tmp/chromadb/queries.jsonl $queries_file

queries = [json.loads(line) for line in open("/tmp/chromadb/queries.jsonl").readlines() if line]
questions = [q["text"] for q in queries]
answers = [q["metadata"]["answer"] for q in queries]

# shuffle the QA pairs order to ensure random prefixes
qas = list(zip(questions, answers))
random.seed(7)
random.shuffle(qas)
questions, answers = list(zip(*qas))

print(questions[:5])
print(answers[:5])
print("Number of questions:", len(questions))

--2024-05-08 01:20:39--  https://huggingface.co/datasets/thinkall/NaturalQuestionsQA/resolve/main/queries.jsonl
Resolving huggingface.co (huggingface.co)... 3.160.150.7, 3.160.150.119, 3.160.150.2, ...
Connecting to huggingface.co (huggingface.co)|3.160.150.7|:443... connected.
HTTP request sent, awaiting response... 

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


200 OK
Length: 1380571 (1.3M) [text/plain]
Saving to: ‘/tmp/chromadb/queries.jsonl’

/tmp/chromadb/queri 100%[===================>]   1.32M  3.85MB/s    in 0.3s    

2024-05-08 01:20:40 (3.85 MB/s) - ‘/tmp/chromadb/queries.jsonl’ saved [1380571/1380571]

('who sang buddy can you spare a dime', "what was hawaii 's primary export to the united states", 'when will the la sagrada familia be finished', 'what does v sign in front of mouth mean', 'who died in the plane crash greys anatomy')
(['Bing Crosby', 'Rudy Vallee'], ['pineapple'], ['2026'], ['signify cunnilingus'], ['Dr. Lexie Grey'])
Number of questions: 6775


In [18]:
from io import StringIO 
import sys

class Capturing(list):
    def __enter__(self):
        self._stdout = sys.stdout
        sys.stdout = self._stringio = StringIO()
        return self
    def __exit__(self, *args):
        self.extend(self._stringio.getvalue().splitlines())
        del self._stringio    # free up some memory
        sys.stdout = self._stdout

In [19]:
import time
from typing import List, Tuple, Union

def process_questions(
        questions: List[str],
        answers: List[str],
        num_questions: int,
        wrapper: Union[RagNoInteractionWrapper, RagWrapper],
) -> Tuple[List[str], List[str], List[str]]:
    retrieve_answers = []
    questions_sample = []
    answers_sample = []
    
    st = time.time()
    for idx, qa_problem in enumerate(questions[:num_questions]):
        if idx % 10 == 0:
            ct = time.time()
            print(f"\nProgress {idx/num_questions*100:.2f}%, Time Used {(ct-st)/3600:.2f} hours\n")
        try:
            with Capturing() as print_output:
                wrapper.retrieve(qa_problem)
            retrieve_answers.append(print_output[-3])
            questions_sample.append(qa_problem)
            answers_sample.append(answers[:num_questions][idx])
        except Exception as e:
            print(e)
            print("Error in problem: ", qa_problem)
    
    print(retrieve_answers[:5])
    print("len(retrieve_answers):", len(retrieve_answers))
    print("len(answers_sample):", len(answers_sample))
    print("len(questions_sample):", len(questions_sample))
    
    return retrieve_answers, questions_sample, answers_sample

In [20]:
# https://qa.fastforwardlabs.com/no%20answer/null%20threshold/bert/distilbert/exact%20match/f1/robust%20predictions/2020/06/09/Evaluating_BERT_on_SQuAD.html#F1
def normalize_text(s):
    """Removing articles and punctuation, and standardizing whitespace are all typical text processing steps."""
    import string, re

    def remove_articles(text):
        regex = re.compile(r"\b(a|an|the)\b", re.UNICODE)
        return re.sub(regex, " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def compute_exact_match(prediction, truth):
    return int(normalize_text(prediction) == normalize_text(truth))

def compute_f1_recall(prediction, truth):
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(truth).split()
    
    # if either the prediction or the truth is no-answer then f1 = 1 if they agree, 0 otherwise
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens), int(pred_tokens == truth_tokens)
    
    common_tokens = set(pred_tokens) & set(truth_tokens)
    
    # if there are no common tokens then f1 = 0
    if len(common_tokens) == 0:
        return 0, 0
    
    prec = len(common_tokens) / len(pred_tokens)
    rec = len(common_tokens) / len(truth_tokens)
    
    return 2 * (prec * rec) / (prec + rec), rec

def get_gold_answers(example):
    """helper function that retrieves all possible true answers from a squad2.0 example"""
    
    gold_answers = [answer["text"] for answer in example.answers if answer["text"]]

    # if gold_answers doesn't exist it's because this is a negative example - 
    # the only correct answer is an empty string
    if not gold_answers:
        gold_answers = [""]
        
    return gold_answers

In [21]:
def evaluate(
        questions: List[str],
        answers: List[str],
        num_questions: int,
        wrapper: Union[RagNoInteractionWrapper, RagWrapper],
) -> None:
    retrieve_answers, questions_sample, answers_sample = process_questions(questions, answers, num_questions, wrapper)

    all_em_scores = []
    all_f1_scores = []
    all_recall_scores = []
    for i in range(len(retrieve_answers)):
        prediction = retrieve_answers[i]
        gold_answers = answers_sample[i]
    
        em_score = max((compute_exact_match(prediction, answer)) for answer in gold_answers)
        f1_score = max((compute_f1_recall(prediction, answer)[0]) for answer in gold_answers)
        recall_score = max((compute_f1_recall(prediction, answer)[1]) for answer in gold_answers)
    
        all_em_scores.append(em_score)
        all_f1_scores.append(f1_score)
        all_recall_scores.append(recall_score)
    
        if i % 100 == 0 or recall_score < 0.3:
            print(f"Question: {questions_sample[i]}")
            print(f"Prediction: {prediction}")
            print(f"True Answers: {gold_answers}")
            print(f"EM: {em_score} \t F1: {f1_score} \t Recall: {recall_score}")
    
    print("=======================================")
    print(f"Average EM: {sum(all_em_scores) / len(all_em_scores)}")
    print(f"Average F1: {sum(all_f1_scores) / len(all_f1_scores)}")
    print(f"Average Recall: {sum(all_recall_scores) / len(all_recall_scores)}")

In [23]:
evaluate(
    questions,
    answers,
    100,
    rag_no_interaction_wrapper,
)


Progress 0.00%, Time Used 0.00 hours


Progress 10.00%, Time Used 0.00 hours


Progress 20.00%, Time Used 0.00 hours


Progress 30.00%, Time Used 0.01 hours


Progress 40.00%, Time Used 0.01 hours


Progress 50.00%, Time Used 0.01 hours


Progress 60.00%, Time Used 0.01 hours


Progress 70.00%, Time Used 0.01 hours


Progress 80.00%, Time Used 0.02 hours


Progress 90.00%, Time Used 0.02 hours

['Bing Crosby and Rudy Vallee.', 'Food and clothing.', '2026', 'It signifies cunnilingus.', "Dr. Lexie Grey died in the plane crash on Grey's Anatomy."]
len(retrieve_answers): 100
len(answers_sample): 100
len(questions_sample): 100
Question: who sang buddy can you spare a dime
Prediction: Bing Crosby and Rudy Vallee.
True Answers: ['Bing Crosby', 'Rudy Vallee']
EM: 0 	 F1: 0.5714285714285715 	 Recall: 1.0
Question: what was hawaii 's primary export to the united states
Prediction: Food and clothing.
True Answers: ['pineapple']
EM: 0 	 F1: 0 	 Recall: 0
Question: where did the term liberal arts 

In [24]:
evaluate(
    questions,
    answers,
    100,
    rag_wrapper,
)


Progress 0.00%, Time Used 0.00 hours


Progress 10.00%, Time Used 0.00 hours


Progress 20.00%, Time Used 0.01 hours


Progress 30.00%, Time Used 0.02 hours


Progress 40.00%, Time Used 0.02 hours


Progress 50.00%, Time Used 0.03 hours


Progress 60.00%, Time Used 0.04 hours


Progress 70.00%, Time Used 0.05 hours


Progress 80.00%, Time Used 0.06 hours


Progress 90.00%, Time Used 0.06 hours

['Bing Crosby and Rudy Vallee sang "Brother, Can You Spare a Dime?"', 'pineapple', '2026.', 'The v sign in front of mouth generally means victory or peace.', "Dr. Lexie Grey died in the plane crash on Grey's Anatomy."]
len(retrieve_answers): 100
len(answers_sample): 100
len(questions_sample): 100
Question: who sang buddy can you spare a dime
Prediction: Bing Crosby and Rudy Vallee sang "Brother, Can You Spare a Dime?"
True Answers: ['Bing Crosby', 'Rudy Vallee']
EM: 0 	 F1: 0.3076923076923077 	 Recall: 1.0
Question: what does v sign in front of mouth mean
Prediction: The v sign in front of mout